# Predict Engine Research

Notebook-first CatBoost regression workflow for the prepared training dataset. The primary artifact is `output/model.cbm`.


In [31]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "src":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root


PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research')

In [32]:
import json
from statistics import median

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, train_test_split

from models.config import default_model_params
from models.regressor import build_regressor, resolve_categorical_columns, resolve_feature_columns
from runtime.device import get_device_report
from tracking.logger import get_logger
from tracking.wandb_tracker import finish_run, log_metrics, start_run, update_summary


## Metric Recap

This notebook trains a **regression** model, not a classifier.

Primary metrics for this task:
- `MAE`: average absolute price error in `price_manwon`, easiest to interpret.
- `RMSE`: penalizes large misses more strongly than MAE.
- `R2`: how much variance the model explains.
- CatBoost train/validation loss is tracked with `RMSE` during fitting.

`precision`, `recall`, `F1`, and `accuracy` are classification metrics, so they are not the right primary metrics here.


In [33]:
data_path = project_root / "data" / "prepared_training.csv"

target_column = "price_manwon"
feature_columns = [
    "brand",
    "model_name",
    "trim_name",
    "major_category",
    "size_score",
    "model_year",
    "vehicle_age_years",
    "color",
    "mileage_km",
]
categorical_columns = [
    "brand",
    "model_name",
    "trim_name",
    "major_category",
    "color",
]

drop_unknown_major_category = True
major_category_unknown_value = "unknown"

output_dir = project_root / "output"
model_path = output_dir / "model.cbm"
metrics_path = output_dir / "metrics.json"
feature_manifest_path = output_dir / "feature_manifest.json"

use_cross_validation = True
n_splits = 5
holdout_valid_size = 0.2
random_seed = 42
max_iterations = 2000
early_stopping_rounds = 300
train_log_period = 25
final_iteration_strategy = "p75"

use_wandb = True
wandb_project = "predict_engine_research"
wandb_entity = "wonbeenlee093-asdf"
wandb_run_name = None
wandb_tags = ["catboost", "prepared_training"]

model_params = default_model_params()
model_params["random_seed"] = random_seed
model_params["iterations"] = max_iterations
model_params["verbose"] = train_log_period
model_params["allow_writing_files"] = False

{
    "data_path": str(data_path),
    "target_column": target_column,
    "feature_columns": feature_columns,
    "categorical_columns": categorical_columns,
    "use_cross_validation": use_cross_validation,
    "n_splits": n_splits,
    "holdout_valid_size": holdout_valid_size,
    "max_iterations": max_iterations,
    "early_stopping_rounds": early_stopping_rounds,
    "train_log_period": train_log_period,
    "final_iteration_strategy": final_iteration_strategy,
    "model_params": model_params,
}


{'data_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/data/prepared_training.csv',
 'target_column': 'price_manwon',
 'feature_columns': ['brand',
  'model_name',
  'trim_name',
  'major_category',
  'size_score',
  'model_year',
  'vehicle_age_years',
  'color',
  'mileage_km'],
 'categorical_columns': ['brand',
  'model_name',
  'trim_name',
  'major_category',
  'color'],
 'use_cross_validation': True,
 'n_splits': 5,
 'holdout_valid_size': 0.2,
 'max_iterations': 2000,
 'early_stopping_rounds': 300,
 'train_log_period': 25,
 'final_iteration_strategy': 'p75',
 'model_params': {'loss_function': 'RMSE',
  'eval_metric': 'RMSE',
  'iterations': 2000,
  'learning_rate': 0.05,
  'depth': 8,
  'verbose': 25,
  'allow_writing_files': False,
  'random_seed': 42}}

In [34]:
logger = get_logger()
device_report = get_device_report()
logger.info("Torch device report: %s", device_report)
device_report


2026-03-18 00:14:43 | INFO | Torch device report: {'torch_installed': True, 'torch_version': '2.10.0', 'preferred_device': 'mps', 'cuda_available': False, 'mps_available': True}


{'torch_installed': True,
 'torch_version': '2.10.0',
 'preferred_device': 'mps',
 'cuda_available': False,
 'mps_available': True}

In [35]:
frame = pd.read_csv(data_path)

if drop_unknown_major_category and "major_category" in frame.columns:
    row_count_before = len(frame)
    frame = frame.loc[
        frame["major_category"].fillna(major_category_unknown_value).ne(major_category_unknown_value)
    ].copy()
    logger.info(
        "Dropped %s rows where major_category == %r",
        row_count_before - len(frame),
        major_category_unknown_value,
    )

feature_columns = resolve_feature_columns(frame, target_column, feature_columns)
categorical_columns = resolve_categorical_columns(frame, feature_columns, categorical_columns)

leakage_columns = [column for column in [target_column] if column in feature_columns]
if leakage_columns:
    raise ValueError(f"target leakage detected in feature columns: {leakage_columns}")

for column in categorical_columns:
    frame[column] = frame[column].astype("string").fillna("__missing__")

X = frame.loc[:, feature_columns].copy()
for column in categorical_columns:
    X[column] = X[column].astype(str)

y = pd.to_numeric(frame[target_column], errors="raise")

logger.info("Loaded %s rows with %s features", len(frame), len(feature_columns))
logger.info("Detected categorical columns: %s", categorical_columns)
logger.info("Target column: %s", target_column)
frame.head()


2026-03-18 00:14:43 | INFO | Dropped 1 rows where major_category == 'unknown'
2026-03-18 00:14:43 | INFO | Loaded 2130 rows with 9 features
2026-03-18 00:14:43 | INFO | Detected categorical columns: ['brand', 'model_name', 'trim_name', 'major_category', 'color']
2026-03-18 00:14:43 | INFO | Target column: price_manwon


,brand,model_name,trim_name,major_category,size_score,model_year,vehicle_age_years,color,mileage_km,price_manwon
0,hyundai,캐스퍼,디 에센셜,suv,0.721501,2023,2.92,초록(연두),5770.0,1780
1,kia,올 뉴 모닝 [JA],프레스티지,hatchback,22.797927,2018,8.17,검정색,46161.0,890
2,kgm,코란도 스포츠,CX7 클럽,truck,32.317073,2015,11.33,회색,250598.0,390
3,hyundai,코나,모던 팝,suv,6.709957,2018,8.58,"파랑(남색,곤색)",140663.0,1150
4,hyundai,아반떼AD,밸류 플러스,sedan,8.033419,2018,8.50,흰색,117863.0,970


In [36]:
run = start_run(
    enabled=use_wandb,
    project=wandb_project,
    entity=wandb_entity,
    name=wandb_run_name,
    tags=wandb_tags,
)

if run is not None:
    run.config.update(
        {
            "data_path": str(data_path),
            "target_column": target_column,
            "feature_columns": feature_columns,
            "categorical_columns": categorical_columns,
            "use_cross_validation": use_cross_validation,
            "n_splits": n_splits,
            "holdout_valid_size": holdout_valid_size,
            "max_iterations": max_iterations,
            "early_stopping_rounds": early_stopping_rounds,
            "train_log_period": train_log_period,
            "final_iteration_strategy": final_iteration_strategy,
            "model_params": model_params,
        }
    )
    logger.info(
        "W&B run started: entity=%s | project=%s | run_id=%s | url=%s",
        getattr(run, "entity", None),
        getattr(run, "project", None),
        getattr(run, "id", None),
        getattr(run, "url", None),
    )
else:
    logger.info("W&B logging disabled for this run")

run


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/iwonbin/.netrc
wandb: Currently logged in as: wonbeenlee093 (wonbeenlee093-asdf) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


2026-03-18 00:14:53 | INFO | W&B run started: entity=wonbeenlee093-asdf | project=predict_engine_research | run_id=7gm7ge3j | url=https://wandb.ai/wonbeenlee093-asdf/predict_engine_research/runs/7gm7ge3j


In [37]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse_value = float(root_mean_squared_error(y_true, y_pred))
    return {
        "mae_price_manwon": float(mean_absolute_error(y_true, y_pred)),
        "rmse_price_manwon": rmse_value,
        "r2_price_manwon": float(r2_score(y_true, y_pred)),
        "valid_loss": rmse_value,
    }


def extract_recommended_iterations(model):
    best_iteration = model.get_best_iteration()
    if best_iteration is not None and int(best_iteration) >= 0:
        return int(best_iteration) + 1
    return int(model.tree_count_)


def resolve_final_iterations(recommended_iterations, strategy):
    if not recommended_iterations:
        return int(model_params["iterations"])
    values = np.asarray(recommended_iterations, dtype=float)
    if strategy == "max":
        return int(values.max())
    if strategy == "p75":
        return int(round(np.percentile(values, 75)))
    return int(round(np.median(values)))


def log_eval_history(run, fold_index, evals_result, step_offset):
    if run is None:
        return 0

    learn_metrics = evals_result.get("learn", {})
    valid_metrics = evals_result.get("validation", {}) or evals_result.get("validation_0", {})
    metric_names = sorted(set(learn_metrics) | set(valid_metrics))
    if not metric_names:
        return 0

    history_length = max(
        [len(values) for values in list(learn_metrics.values()) + list(valid_metrics.values())],
        default=0,
    )
    for idx in range(history_length):
        payload = {"fold": fold_index, "iteration": idx + 1}
        for metric_name in metric_names:
            metric_key = metric_name.lower()
            if metric_key == "rmse":
                if idx < len(learn_metrics.get(metric_name, [])):
                    payload["train_loss"] = float(learn_metrics[metric_name][idx])
                if idx < len(valid_metrics.get(metric_name, [])):
                    payload["valid_loss"] = float(valid_metrics[metric_name][idx])
                continue
            if idx < len(learn_metrics.get(metric_name, [])):
                payload[f"learn_{metric_key}"] = float(learn_metrics[metric_name][idx])
            if idx < len(valid_metrics.get(metric_name, [])):
                payload[f"valid_{metric_key}"] = float(valid_metrics[metric_name][idx])
        log_metrics(run, payload, step=step_offset + idx + 1)
    return history_length


def build_split_iterator(X, y):
    if use_cross_validation:
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_seed)
        for fold_index, (train_index, valid_index) in enumerate(splitter.split(X, y), start=1):
            yield fold_index, train_index, valid_index, f"cv_fold_{fold_index}"
        return

    indices = np.arange(len(X))
    train_index, valid_index = train_test_split(
        indices,
        test_size=holdout_valid_size,
        random_state=random_seed,
        shuffle=True,
    )
    yield 1, train_index, valid_index, "holdout"


In [38]:
fold_results = []
recommended_iterations = []
iteration_step_offset = 0

for fold_index, train_index, valid_index, split_name in build_split_iterator(X, y):
    total_splits = n_splits if use_cross_validation else 1
    logger.info(
        "Starting %s (%s/%s) | max_iterations=%s | patience=%s",
        split_name,
        fold_index,
        total_splits,
        model_params["iterations"],
        early_stopping_rounds,
    )

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]
    y_train = y.iloc[train_index]
    y_valid = y.iloc[valid_index]
    fold_model_params = dict(model_params)
    fold_model_params["use_best_model"] = True
    model = build_regressor(fold_model_params)

    fit_kwargs = {
        "eval_set": (X_valid, y_valid),
        "early_stopping_rounds": early_stopping_rounds,
    }
    if categorical_columns:
        fit_kwargs["cat_features"] = categorical_columns

    model.fit(X_train, y_train, **fit_kwargs)

    predictions = model.predict(X_valid)
    fold_metrics = {
        "fold": fold_index,
        "split_name": split_name,
        **compute_metrics(y_valid, predictions),
        "best_iteration": extract_recommended_iterations(model),
        "tree_count": int(model.tree_count_),
    }
    fold_results.append(fold_metrics)
    recommended_iterations.append(fold_metrics["best_iteration"])

    evals_result = model.get_evals_result()
    logged_history_rows = log_eval_history(run, fold_index, evals_result, iteration_step_offset)
    iteration_step_offset += logged_history_rows

    logger.info("%s metrics: %s", split_name, fold_metrics)
    log_metrics(
        run,
        {
            "fold": fold_index,
            "mae_price_manwon": fold_metrics["mae_price_manwon"],
            "rmse_price_manwon": fold_metrics["rmse_price_manwon"],
            "r2_price_manwon": fold_metrics["r2_price_manwon"],
            "valid_loss": fold_metrics["valid_loss"],
            "best_iteration": fold_metrics["best_iteration"],
            "tree_count": fold_metrics["tree_count"],
        },
        step=iteration_step_offset + fold_index,
    )

fold_results_df = pd.DataFrame(fold_results)
fold_results_df


2026-03-18 00:14:53 | INFO | Starting cv_fold_1 (1/5) | max_iterations=2000 | patience=300


0:	learn: 1319.8590174	test: 1284.9622960	best: 1284.9622960 (0)	total: 3.78ms	remaining: 7.56s
25:	learn: 733.0809796	test: 706.9936705	best: 706.9936705 (25)	total: 81.6ms	remaining: 6.19s
50:	learn: 568.1634023	test: 558.1443880	best: 558.1443880 (50)	total: 183ms	remaining: 6.98s
75:	learn: 509.7991311	test: 522.7869052	best: 522.7869052 (75)	total: 270ms	remaining: 6.84s
100:	learn: 483.3839974	test: 514.3926477	best: 514.1837320 (98)	total: 332ms	remaining: 6.25s
125:	learn: 463.1915954	test: 509.4002097	best: 509.4002097 (125)	total: 399ms	remaining: 5.93s
150:	learn: 445.3548786	test: 507.7954023	best: 507.5988591 (148)	total: 457ms	remaining: 5.59s
175:	learn: 425.9343880	test: 506.6049108	best: 506.3697229 (173)	total: 520ms	remaining: 5.39s
200:	learn: 403.7129190	test: 506.7838285	best: 506.0261795 (190)	total: 601ms	remaining: 5.38s
225:	learn: 381.8722862	test: 506.0127171	best: 505.6645929 (218)	total: 683ms	remaining: 5.36s
250:	learn: 365.3697688	test: 506.7227728	best

2026-03-18 00:14:55 | INFO | cv_fold_1 metrics: {'fold': 1, 'split_name': 'cv_fold_1', 'mae_price_manwon': 254.3131818296476, 'rmse_price_manwon': 503.90890474395263, 'r2_price_manwon': 0.8560143227334761, 'valid_loss': 503.90890474395263, 'best_iteration': 401, 'tree_count': 401}
2026-03-18 00:14:55 | INFO | Starting cv_fold_2 (2/5) | max_iterations=2000 | patience=300


700:	learn: 199.8899092	test: 509.7701065	best: 503.9089047 (400)	total: 2.06s	remaining: 3.83s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 503.9089047
bestIteration = 400

Shrink model to first 401 iterations.
0:	learn: 1326.0952672	test: 1264.5328318	best: 1264.5328318 (0)	total: 2.9ms	remaining: 5.8s
25:	learn: 736.5087700	test: 625.9414475	best: 625.9414475 (25)	total: 70.8ms	remaining: 5.37s
50:	learn: 575.9918252	test: 458.1439897	best: 458.1439897 (50)	total: 139ms	remaining: 5.3s
75:	learn: 520.4898088	test: 425.0125661	best: 425.0125661 (75)	total: 194ms	remaining: 4.92s
100:	learn: 492.0907429	test: 418.4466052	best: 418.4466052 (100)	total: 256ms	remaining: 4.82s
125:	learn: 467.7308752	test: 415.9837217	best: 414.3896788 (119)	total: 340ms	remaining: 5.06s
150:	learn: 447.1020234	test: 411.1016134	best: 410.9264776 (148)	total: 414ms	remaining: 5.08s
175:	learn: 422.6985331	test: 411.0951730	best: 409.4150404 (164)	total: 476ms	remaining: 4.93s
200:	l

2026-03-18 00:14:57 | INFO | cv_fold_2 metrics: {'fold': 2, 'split_name': 'cv_fold_2', 'mae_price_manwon': 264.50542421225856, 'rmse_price_manwon': 403.5549192547418, 'r2_price_manwon': 0.9048374793500689, 'valid_loss': 403.5549192547418, 'best_iteration': 253, 'tree_count': 253}
2026-03-18 00:14:57 | INFO | Starting cv_fold_3 (3/5) | max_iterations=2000 | patience=300


...

1200:	learn: 171.9312247	test: 348.3478296	best: 348.3478296 (1200)	total: 3.17s	remaining: 2.11s
1225:	learn: 169.3391105	test: 349.1109189	best: 348.3478296 (1200)	total: 3.25s	remaining: 2.05s
1250:	learn: 165.6545257	test: 349.0695866	best: 348.3478296 (1200)	total: 3.32s	remaining: 1.99s
1275:	learn: 161.8655352	test: 348.8425696	best: 348.3478296 (1200)	total: 3.4s	remaining: 1.93s
1300:	learn: 158.9812241	test: 348.6459339	best: 348.3478296 (1200)	total: 3.47s	remaining: 1.86s
1325:	learn: 156.7023898	test: 348.8085982	best: 348.3478296 (1200)	total: 3.53s	remaining: 1.8s
1350:	learn: 154.0019469	test: 348.8696979	best: 348.3478296 (1200)	total: 3.61s	remaining: 1.73s
1375:	learn: 150.3133222	test: 348.8487409	best: 348.3478296 (1200)	total: 3.68s	remaining: 1.67s
1400:	learn: 146.8986457	test: 348.7799011	best: 348.3478296 (1200)	total: 3.75s	remaining: 1.6s
1425:	learn: 143.7619325	test: 348.5139194	best: 348.3478296 (1200)	total: 3.89s	remaining: 1.57s
1450:	learn: 141.171996

2026-03-18 00:15:05 | INFO | cv_fold_4 metrics: {'fold': 4, 'split_name': 'cv_fold_4', 'mae_price_manwon': 210.55071057069154, 'rmse_price_manwon': 348.14181935995055, 'r2_price_manwon': 0.9275390040376481, 'valid_loss': 348.14181935995055, 'best_iteration': 1465, 'tree_count': 1465}
2026-03-18 00:15:05 | INFO | Starting cv_fold_5 (5/5) | max_iterations=2000 | patience=300


0:	learn: 1338.0225208	test: 1203.5983690	best: 1203.5983690 (0)	total: 2.43ms	remaining: 4.85s
25:	learn: 748.2605146	test: 574.3258104	best: 574.3258104 (25)	total: 71.1ms	remaining: 5.4s
50:	learn: 594.5374465	test: 429.2490401	best: 429.2490401 (50)	total: 130ms	remaining: 4.98s
75:	learn: 536.1810401	test: 408.0508789	best: 408.0073365 (74)	total: 187ms	remaining: 4.75s
100:	learn: 510.9922301	test: 412.1145535	best: 406.7714604 (87)	total: 262ms	remaining: 4.93s
125:	learn: 492.0990667	test: 414.2066196	best: 406.7714604 (87)	total: 327ms	remaining: 4.87s
150:	learn: 474.8610379	test: 419.0107804	best: 406.7714604 (87)	total: 386ms	remaining: 4.73s
175:	learn: 457.6840656	test: 425.2858459	best: 406.7714604 (87)	total: 444ms	remaining: 4.6s
200:	learn: 436.6547760	test: 422.9270024	best: 406.7714604 (87)	total: 504ms	remaining: 4.51s
225:	learn: 421.5957600	test: 426.3968912	best: 406.7714604 (87)	total: 561ms	remaining: 4.41s
250:	learn: 402.8834434	test: 431.2726396	best: 406.7

2026-03-18 00:15:06 | INFO | cv_fold_5 metrics: {'fold': 5, 'split_name': 'cv_fold_5', 'mae_price_manwon': 269.30905199694786, 'rmse_price_manwon': 406.77146043488466, 'r2_price_manwon': 0.8939267072247857, 'valid_loss': 406.77146043488466, 'best_iteration': 88, 'tree_count': 88}


,fold,split_name,mae_price_manwon,rmse_price_manwon,r2_price_manwon,valid_loss,best_iteration,tree_count
0,1,cv_fold_1,254.313182,503.908905,0.856014,503.908905,401,401
1,2,cv_fold_2,264.505424,403.554919,0.904837,403.554919,253,253
2,3,cv_fold_3,281.153012,893.855103,0.676838,893.855103,406,406
3,4,cv_fold_4,210.550711,348.141819,0.927539,348.141819,1465,1465
4,5,cv_fold_5,269.309052,406.771460,0.893927,406.771460,88,88


In [39]:
aggregate_metrics = {
    "mae_price_manwon": float(fold_results_df["mae_price_manwon"].mean()),
    "rmse_price_manwon": float(fold_results_df["rmse_price_manwon"].mean()),
    "r2_price_manwon": float(fold_results_df["r2_price_manwon"].mean()),
    "valid_loss": float(fold_results_df["valid_loss"].mean()),
}
final_iterations = resolve_final_iterations(recommended_iterations, final_iteration_strategy)

logger.info("Aggregate validation metrics: %s", aggregate_metrics)
logger.info(
    "Recommended final iterations from validation runs: %s (strategy=%s)",
    final_iterations,
    final_iteration_strategy,
)

update_summary(
    run,
    {
        **aggregate_metrics,
        "final_iterations": final_iterations,
        "final_iteration_strategy": final_iteration_strategy,
        "training_rows": int(len(frame)),
        "feature_count": int(len(feature_columns)),
    },
)

{
    "aggregate_metrics": aggregate_metrics,
    "final_iterations": final_iterations,
    "final_iteration_strategy": final_iteration_strategy,
}


2026-03-18 00:15:06 | INFO | Aggregate validation metrics: {'mae_price_manwon': 255.96627621490364, 'rmse_price_manwon': 511.246441402643, 'r2_price_manwon': 0.8518311716237333, 'valid_loss': 511.246441402643}
2026-03-18 00:15:06 | INFO | Recommended final iterations from validation runs: 406 (strategy=p75)


{'aggregate_metrics': {'mae_price_manwon': 255.96627621490364,
  'rmse_price_manwon': 511.246441402643,
  'r2_price_manwon': 0.8518311716237333,
  'valid_loss': 511.246441402643},
 'final_iterations': 406,
 'final_iteration_strategy': 'p75'}

In [40]:
output_dir.mkdir(parents=True, exist_ok=True)

final_model_params = dict(model_params)
final_model_params["iterations"] = final_iterations
final_model_params.pop("use_best_model", None)
final_model = build_regressor(final_model_params)

final_fit_kwargs = {}
if categorical_columns:
    final_fit_kwargs["cat_features"] = categorical_columns

logger.info("Training final model on full dataset with iterations=%s", final_iterations)
final_model.fit(X, y, **final_fit_kwargs)
final_model.save_model(model_path)

metrics_payload = {
    "target_column": target_column,
    "use_cross_validation": use_cross_validation,
    "n_splits": n_splits if use_cross_validation else 1,
    "holdout_valid_size": holdout_valid_size,
    "max_iterations": max_iterations,
    "early_stopping_rounds": early_stopping_rounds,
    "train_log_period": train_log_period,
    "final_iteration_strategy": final_iteration_strategy,
    "model_params": final_model_params,
    "fold_results": fold_results,
    "aggregate_metrics": aggregate_metrics,
    "final_iterations": final_iterations,
    "tree_count": int(final_model.tree_count_),
    "training_rows": int(len(frame)),
}
feature_manifest = {
    "target_column": target_column,
    "feature_columns": feature_columns,
    "categorical_columns": categorical_columns,
}

metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
feature_manifest_path.write_text(json.dumps(feature_manifest, indent=2), encoding="utf-8")

update_summary(
    run,
    {
        "model_path": str(model_path),
        "tree_count": int(final_model.tree_count_),
        "final_iterations": final_iterations,
        "final_iteration_strategy": final_iteration_strategy,
    },
)
finish_run(run)

logger.info("Saved model to %s", model_path)
logger.info("Saved metrics to %s", metrics_path)
logger.info("Saved feature manifest to %s", feature_manifest_path)

{
    "model_path": str(model_path),
    "metrics_path": str(metrics_path),
    "feature_manifest_path": str(feature_manifest_path),
}


2026-03-18 00:15:06 | INFO | Training final model on full dataset with iterations=406


0:	learn: 1310.4075109	total: 3.83ms	remaining: 1.55s
25:	learn: 710.7477914	total: 217ms	remaining: 3.17s
50:	learn: 545.5316522	total: 552ms	remaining: 3.84s
75:	learn: 490.2568375	total: 680ms	remaining: 2.95s
100:	learn: 463.1693317	total: 981ms	remaining: 2.96s
125:	learn: 442.8848377	total: 1.1s	remaining: 2.44s
150:	learn: 421.4962797	total: 1.19s	remaining: 2.02s
175:	learn: 400.7966116	total: 1.27s	remaining: 1.66s
200:	learn: 382.2563877	total: 1.34s	remaining: 1.36s
225:	learn: 367.1031957	total: 1.42s	remaining: 1.13s
250:	learn: 352.3514598	total: 1.51s	remaining: 930ms
275:	learn: 338.0881175	total: 1.59s	remaining: 748ms
300:	learn: 325.4755175	total: 1.68s	remaining: 585ms
325:	learn: 314.4383316	total: 1.75s	remaining: 429ms
350:	learn: 304.6113889	total: 1.81s	remaining: 284ms
375:	learn: 295.1372586	total: 1.89s	remaining: 151ms
400:	learn: 287.4809931	total: 1.97s	remaining: 24.6ms
405:	learn: 286.4291581	total: 1.99s	remaining: 0us


best_iteration,▃▂▃█▁
fold,▁▁▁▁▁▁▁▁▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆█
iteration,▁▁▂▂▂▂▄▄▂▂▂▂▂▃▃▁▂▂▂▃▃▃▄▄▄▆▆▆▆▆▇▇▇██▁▁▁▂▂
mae_price_manwon,▅▆█▁▇
r2_price_manwon,▆▇▁█▇
rmse_price_manwon,▃▂█▁▂
train_loss,▄▄▄▃▃▂▂▂▂▆▂▂█▄▃▃▃▃▃▂▇▆▅▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▄
tree_count,▃▂▃█▁
valid_loss,▅▃▂▂▂▂▂▂▃▂▂▁▁▂▂▂▇▆▆▆▆█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_iteration,88
feature_count,9


2026-03-18 00:15:32 | INFO | Saved model to /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/model.cbm
2026-03-18 00:15:32 | INFO | Saved metrics to /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/metrics.json
2026-03-18 00:15:32 | INFO | Saved feature manifest to /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/feature_manifest.json


{'model_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/model.cbm',
 'metrics_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/metrics.json',
 'feature_manifest_path': '/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/predict_engine_research/output/feature_manifest.json'}